In [2]:

RAIZ = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ))

In [3]:
RAIZ

WindowsPath('c:/Users/ASUS/Desktop/Maestria IA USFQ/AprendizajeSupervisado/Proyectofinal_ml_wti')

In [5]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import feature_extraction as fe
from src import config as cfg

In [7]:
PROC = RAIZ / "data" / "processed"
FIG_DIR = RAIZ / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [9]:
base_ret = pd.read_csv(PROC / "train_validation.csv", index_col=0, parse_dates=True)   # WTI (%), WTI_park (%²)
FOLDS = pd.read_csv(PROC / "folds.csv", parse_dates=["train_ini", "train_fin", "val_ini", "val_fin"])

print(base_ret.shape, base_ret.index.min().date(), "→", base_ret.index.max().date())

(3769, 2) 2010-01-05 → 2024-12-31


In [ ]:
X, y = construir_features(base_ret)

print(f"Variables candidatas: {X.shape[1]}")
print(f"Observaciones: {len(X)} | {X.index.min().date()} → {X.index.max().date()}")
print(f"Días descartados al inicio por falta de historia: {len(base_ret) - len(X)}")
X.head()

In [ ]:
pos = 100
t = X.index[pos]
dia_anterior = base_ret.index[base_ret.index.get_loc(t) - 1]
r_hasta_ayer = base_ret["WTI"].loc[:dia_anterior]

assert np.isclose(X.loc[t, "park_lag1"], base_ret.loc[dia_anterior, "WTI_park"])
assert np.isclose(X.loc[t, "r2_lag1"], base_ret.loc[dia_anterior, "WTI"]**2)
assert np.isclose(X.loc[t, "vol_r_5"], r_hasta_ayer.iloc[-5:].std())
assert np.isclose(X.loc[t, "park_media_21"], base_ret["WTI_park"].loc[:dia_anterior].iloc[-21:].mean())
assert np.isclose(y.loc[t], base_ret.loc[t, "WTI_park"])
print(f"Día τ = {t.date()}: las variables usan información hasta {dia_anterior.date()} ✔")

In [ ]:
grupos = {
    "r2_lag":     ("Rezago del retorno al cuadrado", "%²"),
    "absr_lag":   ("Rezago del retorno en valor absoluto", "%"),
    "park_lag":   ("Rezago del estimador de Parkinson", "%²"),
    "vol_r_":     ("Desviación estándar móvil del retorno", "%"),
    "park_media_": ("Promedio móvil del estimador de Parkinson", "%²"),
    "r_lag1":     ("Retorno del día anterior (con signo)", "%"),
}

def describir(col):
    for pref, (desc, unidad) in grupos.items():
        if col.startswith(pref):
            detalle = col.replace(pref, "")
            if "lag" in pref and pref != "r_lag1":
                detalle = f"τ-{detalle}"
            elif pref != "r_lag1":
                detalle = f"ventana de {detalle} días"
            else:
                detalle = "τ-1"
            return pd.Series({"descripción": desc, "detalle": detalle, "unidad": unidad})

diccionario = X.columns.to_series().apply(describir)
display(diccionario)
display(diccionario["descripción"].value_counts().to_frame("n variables"))

In [ ]:
f1 = FOLDS.iloc[0]
tr = (X.index >= f1.train_ini) & (X.index <= f1.train_fin)

# Correlación de Spearman con la etiqueta (relación monótona, robusta a valores extremos)
corr_y = X[tr].corrwith(y[tr], method="spearman").sort_values()
corr_y.plot.barh(figsize=(8, 9), title="Correlación de Spearman con Parkinson del día τ (train fold 1)")
plt.tight_layout(); plt.savefig(FIG_DIR / "03b_correlacion_features_y.png", dpi=150); plt.show()

# Redundancia entre variables
corr_X = X[tr].corr(method="spearman")
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr_X, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_X)), corr_X.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(corr_X)), corr_X.columns, fontsize=7)
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Correlación de Spearman entre variables candidatas (train fold 1)")
plt.tight_layout(); plt.savefig(FIG_DIR / "03b_correlacion_entre_features.png", dpi=150); plt.show()

In [ ]:
X.to_csv(PROC / "features_X.csv")
y.to_csv(PROC / "features_y.csv")
diccionario.to_csv(PROC / "features_diccionario.csv")
print("Guardado en", PROC)